# 03 — Modelo: o crédito vai ganhar força no próximo trimestre?

**A pergunta** (seção 6.1 do `docs/architecture.md`): para cada combinação estado × modalidade, *o saldo de crédito vai crescer **mais** no próximo trimestre do que cresceu nos últimos 3 meses?*

- **Ganhar força** = crescer mais rápido do que vinha crescendo → sinal para **expandir** (classe positiva).
- **Perder força** = continuar crescendo mais devagar, ou cair → **cuidado ao expandir**.
- O "próximo trimestre" é de **t+2 a t+5**: o BCB publica o mês t uns 60 dias depois, então t+1 e t+2 já passaram quando o dado chega ao decisor.
- Vale para as **174 combinações** com os 10 anos completos (99,995% do saldo).

*Por que não "vai crescer mais que o Brasil?"* — essa foi a primeira versão. Ao comparar cada estado com a média do país, o efeito da Selic (igual para todos) se cancela. A pergunta atual mantém o ciclo do crédito, onde a Selic atua.

Este notebook **só lê** os resultados que o pipeline gravou em `data/final/` (rode `python scripts/run_pipeline.py` antes).

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src import config
from src.analise.correlacao import nome_curto

r = json.loads(config.ARQUIVO_ML_RESULTADOS.read_text(encoding="utf-8"))
producao = pd.read_parquet(config.ARQUIVO_ML_PREVISAO_PRODUCAO)
print("Linhas por conjunto:", r["linhas_por_conjunto"])
print("Combinações fora da coorte:", len(r["combinacoes_excluidas"]))
print("Modelo escolhido na janela móvel:", r["modelo_escolhido"])

Linhas por conjunto: {'desenvolvimento': 14964, 'teste': 2088, 'embargo': 870, 'producao': 174}
Combinações fora da coorte: 42
Modelo escolhido na janela móvel: gradient_boosting


## 1. O modelo contra os palpites simples (teste final, fev/2025–jan/2026)

O teste final foi usado **uma única vez**. As duas réguas:
- **classe majoritária**: chutar sempre a resposta mais comum — equivale a jogar moeda (AUC 0,5);
- **volta ao normal**: "se perdeu força nos últimos 3 meses, vai ganhar" — a régua de verdade.

**Como ler:** AUC mede se o modelo põe à frente quem de fato ganhou força (0,5 = moeda; 1 = perfeito). *Melhores apostas* = das 20 combinações que o modelo mais recomendou expandir em cada mês, quantas de fato ganharam força.

In [2]:
colunas = ["auc", "auc_pesada_volume", "precisao_melhores_apostas", "precisao", "recall", "acuracia"]
tabela = pd.DataFrame({nome: {c: r["teste"][nome][c] for c in colunas}
                       for nome in ["modelo", "modelo_sem_selic", "volta_ao_normal", "classe_majoritaria"]}).T
tabela.round(3)

,auc,auc_pesada_volume,precisao_melhores_apostas,precisao,recall,acuracia
modelo,0.778,0.792,0.825,0.677,0.827,0.715
modelo_sem_selic,0.796,0.808,0.867,0.680,0.840,0.722
volta_ao_normal,0.688,0.701,0.750,0.654,0.656,0.654
classe_majoritaria,0.500,0.500,0.425,0.501,1.000,0.501


In [3]:
s = r["resumo"]
print(f"O modelo supera a volta ao normal? {'SIM' if s['supera_volta_ao_normal'] else 'NÃO'} "
      f"(AUC {s['auc_modelo']:.3f} contra {s['auc_volta_ao_normal']:.3f})")
print(f"Melhores apostas: {s['melhores_apostas_modelo']:.1%} de acerto, contra {s['melhores_apostas_volta_ao_normal']:.1%} da volta ao normal")
print("Alertas de métrica alta demais (> 0,95):", r["alertas"] or "nenhum")

O modelo supera a volta ao normal? SIM (AUC 0.778 contra 0.688)
Melhores apostas: 82.5% de acerto, contra 75.0% da volta ao normal
Alertas de métrica alta demais (> 0,95): nenhum


## 2. O resultado se repete em anos diferentes? (janela móvel)

Cada ano de 2020 a 2023 foi previsto por um modelo treinado só com o passado (até julho do ano anterior). Foi aqui que o modelo foi escolhido — sem olhar o teste final. Os anos cobrem juros em queda (2020), na mínima (2021) e em alta (2022–23).

In [4]:
pd.DataFrame(r["janela_movel"]).drop(["linhas_treino", "linhas_avaliacao"]).round(3)

,2020,2021,2022,2023
regressao_logistica,0.626,0.748,0.788,0.726
regressao_logistica_sem_selic,0.684,0.698,0.734,0.741
gradient_boosting,0.757,0.757,0.732,0.763
gradient_boosting_sem_selic,0.750,0.722,0.799,0.797
classe_majoritaria,0.500,0.500,0.500,0.500
volta_ao_normal,0.669,0.661,0.714,0.685


## 3. A Selic ajuda a prever?

O mesmo modelo foi treinado **com** e **sem** as variáveis da Selic (nível da meta e decisões do Copom nos 3 e 6 meses anteriores). A diferença de AUC ("com − sem") mostra o quanto a Selic agrega.

In [5]:
efeito = pd.Series(s["efeito_selic_auc_por_bloco"], name="com − sem Selic (AUC)")
efeito["teste final"] = s["efeito_selic_auc_teste"]
efeito.round(3).to_frame()

,com − sem Selic (AUC)
2020,0.007
2021,0.035
2022,-0.067
2023,-0.034
teste final,-0.017


**Leitura:** a diferença oscila em torno de zero — ora um pouco positiva, ora negativa — e no teste final o modelo **sem** a Selic foi ligeiramente melhor. Conclusão: **a Selic não ajuda a prever se o crédito de um estado vai ganhar força no próximo trimestre.** O que prevê é a dinâmica do próprio crédito (seção 5 abaixo).

Isso **não contradiz** a Sprint 4: lá a Selic aparece **associada** ao crédito (andam juntos); aqui a pergunta é se ela **avisa antes**, no horizonte de um trimestre — e não avisa. Para o decisor, o recado é: a decisão trimestral pode se apoiar no comportamento recente do crédito de cada estado, sem depender de prever o Copom.

## 4. Por modalidade

In [6]:
por_mod = pd.DataFrame({
    "modelo": r["teste"]["modelo"]["auc_por_modalidade"],
    "volta_ao_normal": r["teste"]["volta_ao_normal"]["auc_por_modalidade"],
}).rename(index=nome_curto)
por_mod.assign(ganho=lambda d: d["modelo"] - d["volta_ao_normal"]).sort_values("modelo", ascending=False).round(3)

,modelo,volta_ao_normal,ganho
À importação,0.889,0.760,0.129
Financiamentos (geral),0.859,0.713,0.146
De títulos e valores mobiliários,0.853,0.736,0.117
À exportação,0.840,0.718,0.122
Rurais,0.769,0.662,0.107
Imobiliários,0.752,0.702,0.050
De infraestrutura e desenvolvimento,0.730,0.725,0.004
Com interveniência,0.688,0.672,0.016


## 5. O que o modelo usa

Importância por permutação: quanto a AUC cai quando os valores de uma variável são embaralhados. Quanto maior, mais o modelo depende dela.

In [7]:
pd.Series(r["teste"]["importancia_variaveis"], name="queda de AUC").round(4).to_frame().head(10)

,queda de AUC
var_3m,0.1700
var_6m,0.0193
aceleracao_3m,0.0163
var_12m,0.0089
selic_var_3m,0.0067
participacao,0.0062
rel_3m,0.0025
rel_12m,0.0014
uf,0.0007
var_participacao_12m,0.0006


A variável dominante é o **crescimento dos últimos 3 meses** (`var_3m`): quem cresceu muito tende a desacelerar, e quem cresceu pouco tende a acelerar — é a "volta ao normal", que o modelo refina com o crescimento de 6 e 12 meses e a aceleração. As variáveis da Selic aparecem com peso pequeno.

## 6. A previsão para set–nov/2026

Probabilidade de cada combinação **ganhar força** no trimestre set–nov/2026, a partir do que se sabia em jun/2026 (t0). É a matéria-prima da decisão da Sprint 6, onde se escolhe o limiar pelo custo do erro (seção 10).

⚠ As probabilidades mais extremas aparecem em combinações **pequenas** (exportação e importação em estados menores), que oscilam muito. Por isso a métrica pesada pelo volume e a decisão da Sprint 6 importam.

In [8]:
prod = producao.assign(modalidade=producao["modalidade"].map(nome_curto))
print("10 com MAIOR chance de ganhar força:")
display(prod.head(10)[["uf", "modalidade", "prob_ganha_forca"]].round(3))
print("10 com MENOR chance (mais propensas a perder força):")
display(prod.tail(10)[["uf", "modalidade", "prob_ganha_forca"]].round(3))

10 com MAIOR chance de ganhar força:


,uf,modalidade,prob_ganha_forca
0,AP,À exportação,0.953
1,PI,À importação,0.932
2,RN,À importação,0.929
3,RO,À exportação,0.926
4,RO,Rurais,0.925
5,TO,Rurais,0.922
6,GO,Rurais,0.920
7,MS,Rurais,0.919
8,RS,Rurais,0.919
9,MA,Rurais,0.917


10 com MENOR chance (mais propensas a perder força):


,uf,modalidade,prob_ganha_forca
164,PA,De infraestrutura e desenvolvimento,0.068
165,MT,À importação,0.067
166,PI,Com interveniência,0.061
167,TO,Financiamentos (geral),0.058
168,GO,De títulos e valores mobiliários,0.056
169,MG,De títulos e valores mobiliários,0.044
170,GO,À importação,0.029
171,ES,De títulos e valores mobiliários,0.029
172,SP,De títulos e valores mobiliários,0.027
173,PR,De títulos e valores mobiliários,0.011


## 7. Por que os números parecem mais firmes do que são

- **Rótulos sobrepostos:** a previsão feita em janeiro e a feita em fevereiro compartilham 2 dos 3 meses do trimestre futuro — as linhas não são independentes.
- **Choques comuns:** as 174 combinações do mesmo mês sofrem os mesmos choques (juros, economia).

Por isso não calculamos margem de erro sobre as ~2 mil linhas do teste como se fossem independentes. A incerteza real aparece na **variação entre os 4 anos** da janela móvel (seção 2).

## 8. Checklist anti-vazamento (seção 6.3)

| Item | Resposta | Evidência |
|---|---|---|
| Toda feature existia antes do t0? | **Sim.** Todas as features de uma origem t usam só meses ≤ t, e a produção usa só dados até jun/2026 | `src/ml/dataset.py` (só `shift` para trás nas features); teste `test_mexer_no_futuro_nao_muda_as_features` |
| As agregações (lags, médias móveis) usaram só dados anteriores ao t0 de cada observação? | **Sim.** Variações, aceleração, participação e Selic são calculadas com `shift(k)` para trás a partir da própria origem | mesmo teste acima: alterar volume e Selic depois da origem não muda nenhuma feature |
| O split respeita a ordem temporal? Nenhum mês de teste antes de um de treino? | **Sim**, com **embargo de 5 meses** (set/2024–jan/2025), porque o rótulo olha até t+5; na janela móvel, cada ano é treinado só até julho do ano anterior | `marcar_conjunto` e `janela_movel`; teste `test_conjuntos_sem_sobreposicao_e_chave_unica` |
| Scalers e encoders ajustados só no treino? | **Sim.** Pré-processamento e modelo num único `Pipeline`, com `fit` só nas linhas de treino de cada avaliação | `montar_pipeline` em `src/ml/treino.py`; teste `test_padronizacao_ajustada_so_no_treino` |
| A coluna que origina o label saiu das features? | **Sim.** O volume futuro (t+2 e t+5) só aparece no rótulo; a quantidade de operações nem entra | lista `FEATURES_NUMERICAS` em `src/ml/dataset.py` |

Verificações extras: a escolha do modelo não muda se os rótulos do teste forem invertidos (`test_rotulos_do_teste_nao_mudam_a_escolha`), e métricas acima de 0,95 geram alerta (`test_base_perfeita_dispara_alerta`) — na execução real, nenhum alerta.